In [1]:
import sys
import subprocess

packages = [
    "transformers==5.24.0",
    "huggingface-hub==1.33.0",
    "accelerate==1.15.0",
    "datasets==4.8.5",
    "pandas==2.2.3",
    "scikit-learn==1.6.1",
    "jedi==0.19.2"
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True
)

import transformers
import huggingface_hub

print("Transformers loaded:", transformers.__version__)
print("Hub loaded:", huggingface_hub.__version__)

assert transformers.__version__ == "5.24.0"
assert huggingface_hub.__version__ == "1.33.0"

print("Environment imports are consistent.")

Transformers loaded: 5.24.0
Hub loaded: 1.33.0
Environment imports are consistent.


In [2]:
import random
import numpy as np
import pandas as pd
import torch

from importlib.metadata import version

SEED = 42

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda")
gpu = torch.cuda.get_device_properties(device)

print(f"Python: {sys.version.split()[0]}")
print(f"GPU: {gpu.name}")
print(f"GPU memory: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Random seed: {SEED}")

for package in ["torch", "numpy", "pandas", "datasets", "scikit-learn", "accelerate"]:
    print(f"{package}: {version(package)}")

Python: 3.13.16
GPU: Tesla T4
GPU memory: 14.6 GB
Random seed: 42
torch: 2.14.1+cu130
numpy: 2.4.6
pandas: 2.2.3
datasets: 4.8.5
scikit-learn: 1.6.1
accelerate: 1.15.0


In [3]:
import io
import json

from zipfile import ZipFile
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Upload only m3_data_setup.zip."

archive_name, archive_bytes = next(iter(uploaded.items()))

with ZipFile(io.BytesIO(archive_bytes)) as archive:
    required_files = {
        "split_manifest.json",
        "split_counts.csv",
        "split_overlap.csv"
    }
    assert required_files.issubset(archive.namelist())

    split_manifest = json.loads(archive.read("split_manifest.json"))
    saved_split_counts = pd.read_csv(archive.open("split_counts.csv"))
    saved_split_overlap = pd.read_csv(archive.open("split_overlap.csv"))

DATASET_NAME = split_manifest["dataset"]
DATASET_REVISION = split_manifest["dataset_revision"]
LABELS = split_manifest["labels"]

assert DATASET_NAME == "cnamuangtoun/resume-job-description-fit"
assert split_manifest["seed"] == SEED
assert LABELS == ["Good Fit", "Potential Fit", "No Fit"]

LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}

print(f"Archive: {archive_name}")
print(f"Dataset revision: {DATASET_REVISION}")
print(f"Label mapping: {LABEL2ID}")

display(saved_split_counts)

Saving m3_data_setup.zip to m3_data_setup.zip
Archive: m3_data_setup.zip
Dataset revision: 08978e21714984bb417547d2c0f9b477f5298163
Label mapping: {'Good Fit': 0, 'Potential Fit': 1, 'No Fit': 2}


,Split,Good Fit,Potential Fit,No Fit,Total,Unique jobs
0,Training,1258,1223,2600,5081,229
1,Validation,280,331,536,1147,51
2,Official test,458,444,857,1759,71


In [4]:
from datasets import load_dataset

REQUIRED_COLUMNS = ["resume_text", "job_description_text", "label"]

dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)

source_frames = {
    name: dataset[name].to_pandas()[REQUIRED_COLUMNS]
    for name in ["train", "test"]
}

split_frames = {}

for split_name, row_ids in split_manifest["source_row_ids"].items():
    source_name = split_manifest["source_splits"][split_name]
    source_df = source_frames[source_name]

    assert len(row_ids) == len(set(row_ids))
    assert all(0 <= row_id < len(source_df) for row_id in row_ids)

    dataframe = source_df.iloc[row_ids].copy()
    dataframe["source_row_id"] = row_ids
    split_frames[split_name] = dataframe.reset_index(drop=True)

train_df = split_frames["Training"]
validation_df = split_frames["Validation"]
test_df = split_frames["Official test"]

actual_split_counts = pd.DataFrame({
    name: dataframe["label"].value_counts()
    for name, dataframe in split_frames.items()
}).reindex(LABELS).fillna(0).astype(int).T

actual_split_counts["Total"] = actual_split_counts.sum(axis=1)
expected_counts = saved_split_counts.set_index("Split")[LABELS + ["Total"]]

display(actual_split_counts)
assert actual_split_counts.equals(expected_counts)

print("All splits match the saved row and class counts.")

train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

label,Good Fit,Potential Fit,No Fit,Total
Training,1258,1223,2600,5081
Validation,280,331,536,1147
Official test,458,444,857,1759


All splits match the saved row and class counts.


In [5]:
import re
import hashlib
import unicodedata

EMAIL_PATTERN = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
URL_PATTERN = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
PHONE_PATTERN = re.compile(r"(?<!\w)(?:\+?\d[\d\s().-]{8,}\d)(?!\w)")


def prepare_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = EMAIL_PATTERN.sub(" email ", text)
    text = URL_PATTERN.sub(" url ", text)
    text = PHONE_PATTERN.sub(
        lambda match: " phone "
        if sum(character.isdigit() for character in match.group()) >= 10
        else match.group(),
        text
    )
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[^\S\n]+", " ", text)
    return re.sub(r"\n\s*\n+", "\n", text).strip()


def text_group_id(text):
    normalized = " ".join(text.casefold().split())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


assert prepare_text("Skills: C++, C#, .NET") == "Skills: C++, C#, .NET"
print("Text preparation functions are ready.")

Text preparation functions are ready.


In [6]:
def prepare_frame(dataframe):
    result = dataframe.copy()
    result["resume_model_text"] = result["resume_text"].map(prepare_text)
    result["job_model_text"] = result["job_description_text"].map(prepare_text)
    result["resume_group"] = result["resume_model_text"].map(text_group_id)
    result["job_group"] = result["job_model_text"].map(text_group_id)
    result["labels"] = result["label"].map(LABEL2ID)

    assert result["labels"].notna().all()
    result["labels"] = result["labels"].astype("int64")
    return result


split_frames = {
    name: prepare_frame(dataframe)
    for name, dataframe in split_frames.items()
}

train_df = split_frames["Training"]
validation_df = split_frames["Validation"]
test_df = split_frames["Official test"]

MODEL_PAIR_COLUMNS = ["resume_group", "job_group"]
preparation_rows = []

for name, dataframe in split_frames.items():
    preparation_rows.append({
        "Split": name,
        "Rows": len(dataframe),
        "Unique jobs": dataframe["job_group"].nunique(),
        "Empty resumes": int(dataframe["resume_model_text"].eq("").sum()),
        "Empty jobs": int(dataframe["job_model_text"].eq("").sum()),
        "Duplicate pairs": int(dataframe.duplicated(MODEL_PAIR_COLUMNS).sum())
    })
    assert set(dataframe["labels"]) == set(ID2LABEL)

preparation_report = pd.DataFrame(preparation_rows)
display(preparation_report)

assert preparation_report[
    ["Empty resumes", "Empty jobs", "Duplicate pairs"]
].to_numpy().sum() == 0

assert preparation_report.set_index("Split")["Unique jobs"].equals(
    saved_split_counts.set_index("Split")["Unique jobs"]
)

,Split,Rows,Unique jobs,Empty resumes,Empty jobs,Duplicate pairs
0,Training,5081,229,0,0,0
1,Validation,1147,51,0,0,0
2,Official test,1759,71,0,0,0


In [7]:
from itertools import combinations

overlap_rows = []

for (first_name, first_df), (second_name, second_df) in combinations(
    split_frames.items(), 2
):
    first_pairs = set(
        first_df[MODEL_PAIR_COLUMNS].itertuples(index=False, name=None)
    )
    second_pairs = set(
        second_df[MODEL_PAIR_COLUMNS].itertuples(index=False, name=None)
    )

    overlap_rows.append({
        "Comparison": f"{first_name} / {second_name}",
        "Shared jobs": len(
            set(first_df["job_group"]) & set(second_df["job_group"])
        ),
        "Shared resumes": len(
            set(first_df["resume_group"]) & set(second_df["resume_group"])
        ),
        "Shared pairs": len(first_pairs & second_pairs)
    })

split_overlap_report = pd.DataFrame(overlap_rows)
display(split_overlap_report)

assert split_overlap_report.equals(saved_split_overlap)
assert split_overlap_report["Shared jobs"].eq(0).all()
assert split_overlap_report["Shared pairs"].eq(0).all()

print("Reconstructed splits match Vikas's saved overlap report exactly.")

,Comparison,Shared jobs,Shared resumes,Shared pairs
0,Training / Validation,0,429,0
1,Training / Official test,0,473,0
2,Validation / Official test,0,357,0


Reconstructed splits match Vikas's saved overlap report exactly.


In [8]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from huggingface_hub import HfApi

MODEL_CHECKPOINTS = {
    "MiniLM": "microsoft/MiniLM-L12-H384-uncased",
    "DistilBERT": "distilbert/distilbert-base-uncased"
}

MAX_LENGTH = 512
tokenizers = {}
model_revisions = {}
tokenizer_rows = []

api = HfApi()

for name, checkpoint in MODEL_CHECKPOINTS.items():
    revision = api.model_info(checkpoint).sha
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, revision=revision)

    assert tokenizer.pad_token_id is not None
    assert MAX_LENGTH <= tokenizer.model_max_length

    tokenizers[name] = tokenizer
    model_revisions[name] = revision

    tokenizer_rows.append({
        "Model": name,
        "Tokenizer": type(tokenizer).__name__,
        "Vocabulary size": len(tokenizer),
        "Model input limit": tokenizer.model_max_length,
        "Pair special tokens": tokenizer.num_special_tokens_to_add(pair=True)
    })

display(pd.DataFrame(tokenizer_rows))

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

,Model,Tokenizer,Vocabulary size,Model input limit,Pair special tokens
0,MiniLM,BertTokenizer,30522,1000000000000000019884624838656,3
1,DistilBERT,BertTokenizer,30522,512,3


In [9]:
from transformers import AutoConfig

model_configs = {}
input_limit_rows = []

for name, checkpoint in MODEL_CHECKPOINTS.items():
    config = AutoConfig.from_pretrained(
        checkpoint,
        revision=model_revisions[name]
    )

    actual_limit = config.max_position_embeddings
    assert MAX_LENGTH <= actual_limit

    model_configs[name] = config
    tokenizers[name].model_max_length = actual_limit

    input_limit_rows.append({
        "Model": name,
        "Architecture": config.model_type,
        "Actual input limit": actual_limit,
        "Selected input limit": MAX_LENGTH,
        "Uses token type IDs": config.model_type == "bert"
    })

display(pd.DataFrame(input_limit_rows))

,Model,Architecture,Actual input limit,Selected input limit,Uses token type IDs
0,MiniLM,bert,512,512,True
1,DistilBERT,distilbert,512,512,False


In [10]:
unique_texts = pd.concat([
    split_frames[split_name][column]
    for split_name in ["Training", "Validation"]
    for column in ["resume_model_text", "job_model_text"]
]).drop_duplicates().tolist()

length_data = {}
length_rows = []

for name, tokenizer in tokenizers.items():
    encoded = tokenizer(
        unique_texts,
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False
    )

    text_lengths = {
        text: len(ids)
        for text, ids in zip(unique_texts, encoded["input_ids"])
    }
    del encoded

    length_data[name] = {}

    for split_name in ["Training", "Validation"]:
        dataframe = split_frames[split_name]
        resume_lengths = dataframe["resume_model_text"].map(text_lengths).to_numpy()
        job_lengths = dataframe["job_model_text"].map(text_lengths).to_numpy()
        pair_lengths = (
            resume_lengths + job_lengths
            + tokenizer.num_special_tokens_to_add(pair=True)
        )

        length_data[name][split_name] = {
            "resume": resume_lengths,
            "job": job_lengths,
            "pair": pair_lengths
        }

        length_rows.append({
            "Model": name,
            "Split": split_name,
            "Mean resume tokens": round(resume_lengths.mean(), 1),
            "Mean job tokens": round(job_lengths.mean(), 1),
            "Median pair tokens": round(float(np.median(pair_lengths)), 1),
            "Pairs over 512 (%)": round(float((pair_lengths > MAX_LENGTH).mean() * 100), 2)
        })

length_report = pd.DataFrame(length_rows)
display(length_report)

,Model,Split,Mean resume tokens,Mean job tokens,Median pair tokens,Pairs over 512 (%)
0,MiniLM,Training,1153.7,530.9,1576.0,99.63
1,MiniLM,Validation,1169.9,404.0,1443.0,99.39
2,DistilBERT,Training,1153.7,530.9,1576.0,99.63
3,DistilBERT,Validation,1169.9,404.0,1443.0,99.39


In [11]:
from datasets import Dataset, DatasetDict

TRUNCATION_POLICY = "longest_first"


def tokenize_pairs(batch, tokenizer, use_token_type_ids):
    return tokenizer(
        batch["resume_model_text"],
        batch["job_model_text"],
        truncation=TRUNCATION_POLICY,
        max_length=MAX_LENGTH,
        padding=False,
        return_attention_mask=True,
        return_token_type_ids=use_token_type_ids
    )


tokenized_datasets = {}

for name, tokenizer in tokenizers.items():
    tokenizer.truncation_side = "right"
    tokenized_datasets[name] = DatasetDict()

    for split_name, dataframe in split_frames.items():
        source_dataset = Dataset.from_pandas(
            dataframe[["resume_model_text", "job_model_text", "labels"]],
            preserve_index=False
        )

        tokenized_datasets[name][split_name] = source_dataset.map(
            tokenize_pairs,
            batched=True,
            batch_size=128,
            fn_kwargs={
                "tokenizer": tokenizer,
                "use_token_type_ids": model_configs[name].model_type == "bert"
            },
            remove_columns=["resume_model_text", "job_model_text"],
            desc=f"{name}: {split_name}"
        )

    print(name, {
        split: data.num_rows
        for split, data in tokenized_datasets[name].items()
    })
    print("Columns:", tokenized_datasets[name]["Training"].column_names)

MiniLM: Training:   0%|          | 0/5081 [00:00<?, ? examples/s]

MiniLM: Validation:   0%|          | 0/1147 [00:00<?, ? examples/s]

MiniLM: Official test:   0%|          | 0/1759 [00:00<?, ? examples/s]

MiniLM {'Training': 5081, 'Validation': 1147, 'Official test': 1759}
Columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


DistilBERT: Training:   0%|          | 0/5081 [00:00<?, ? examples/s]

DistilBERT: Validation:   0%|          | 0/1147 [00:00<?, ? examples/s]

DistilBERT: Official test:   0%|          | 0/1759 [00:00<?, ? examples/s]

DistilBERT {'Training': 5081, 'Validation': 1147, 'Official test': 1759}
Columns: ['labels', 'input_ids', 'attention_mask']


In [12]:
tokenization_rows = []

for name, model_splits in tokenized_datasets.items():
    expected_columns = {"labels", "input_ids", "attention_mask"}

    if model_configs[name].model_type == "bert":
        expected_columns.add("token_type_ids")

    special_tokens = tokenizers[name].num_special_tokens_to_add(pair=True)

    for split_name, data in model_splits.items():
        input_lengths = np.array([len(ids) for ids in data["input_ids"]])
        mask_lengths = np.array([len(mask) for mask in data["attention_mask"]])

        assert set(data.column_names) == expected_columns
        assert len(data) == len(split_frames[split_name])
        assert np.array_equal(input_lengths, mask_lengths)
        assert input_lengths.min() > special_tokens
        assert input_lengths.max() <= MAX_LENGTH
        assert np.array_equal(
            np.array(data["labels"]),
            split_frames[split_name]["labels"].to_numpy()
        )

        if "token_type_ids" in expected_columns:
            assert np.array_equal(
                input_lengths,
                np.array([len(ids) for ids in data["token_type_ids"]])
            )

        retention = None
        if split_name in length_data[name]:
            original_lengths = length_data[name][split_name]["pair"]
            retention = round(float(np.mean(
                (input_lengths - special_tokens)
                / (original_lengths - special_tokens) * 100
            )), 2)

        tokenization_rows.append({
            "Model": name,
            "Split": split_name,
            "Rows": len(data),
            "Minimum tokens": int(input_lengths.min()),
            "Maximum tokens": int(input_lengths.max()),
            "Mean tokens retained (%)": retention
        })

tokenization_report = pd.DataFrame(tokenization_rows)
display(tokenization_report)

,Model,Split,Rows,Minimum tokens,Maximum tokens,Mean tokens retained (%)
0,MiniLM,Training,5081,334,512,34.76
1,MiniLM,Validation,1147,437,512,37.84
2,MiniLM,Official test,1759,409,512,NaN
3,DistilBERT,Training,5081,334,512,34.76
4,DistilBERT,Validation,1147,437,512,37.84
5,DistilBERT,Official test,1759,409,512,NaN


In [13]:
collators = {}
batch_rows = []

for name, tokenizer in tokenizers.items():
    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt"
    )
    collators[name] = collator

    training_data = tokenized_datasets[name]["Training"]
    lengths = np.array([len(ids) for ids in training_data["input_ids"]])
    sorted_indices = np.argsort(lengths)
    sample_indices = np.concatenate(
        [sorted_indices[:4], sorted_indices[-4:]]
    ).tolist()

    features = [training_data[index] for index in sample_indices]
    batch = collator(features)
    padding_positions = batch["attention_mask"].eq(0)

    assert set(batch) == set(features[0])
    assert all(tensor.dtype == torch.long for tensor in batch.values())
    assert batch["input_ids"].shape == batch["attention_mask"].shape
    assert batch["labels"].tolist() == [item["labels"] for item in features]
    assert batch["attention_mask"].sum(dim=1).tolist() == [
        len(item["input_ids"]) for item in features
    ]
    assert padding_positions.any().item()
    assert batch["input_ids"][padding_positions].eq(tokenizer.pad_token_id).all().item()

    batch_rows.append({
        "Model": name,
        "Input shape": str(tuple(batch["input_ids"].shape)),
        "Padding tokens": int(padding_positions.sum().item()),
        "Label dtype": str(batch["labels"].dtype)
    })

batch_report = pd.DataFrame(batch_rows)
display(batch_report)

,Model,Input shape,Padding tokens,Label dtype
0,MiniLM,"(8, 512)",475,torch.int64
1,DistilBERT,"(8, 512)",475,torch.int64


In [14]:
from pathlib import Path
from importlib.metadata import version

OUTPUT_DIR = Path("/content/m3_tokenization")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tokenization_config = {
    "dataset": DATASET_NAME,
    "dataset_revision": DATASET_REVISION,
    "python_version": sys.version.split()[0],
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "truncation": TRUNCATION_POLICY,
    "truncation_side": "right",
    "padding": "dynamic",
    "pad_to_multiple_of": 8,
    "pair_order": ["resume_model_text", "job_model_text"],
    "label2id": LABEL2ID,
    "checkpoints": MODEL_CHECKPOINTS,
    "model_revisions": model_revisions,
    "token_type_ids": {
        name: config.model_type == "bert"
        for name, config in model_configs.items()
    },
    "package_versions": {
        package: version(package)
        for package in [
            "transformers", "huggingface-hub", "tokenizers", "accelerate",
            "datasets", "numpy", "pandas", "scikit-learn", "torch"
        ]
    }
}

(OUTPUT_DIR / "tokenization_config.json").write_text(
    json.dumps(tokenization_config, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2), encoding="utf-8"
)

length_report.to_csv(OUTPUT_DIR / "input_lengths.csv", index=False)
tokenization_report.to_csv(OUTPUT_DIR / "tokenization_report.csv", index=False)
batch_report.to_csv(OUTPUT_DIR / "batch_report.csv", index=False)

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size:,} bytes")

batch_report.csv: 117 bytes
input_lengths.csv: 265 bytes
split_manifest.json: 94,346 bytes
tokenization_config.json: 1,076 bytes
tokenization_report.csv: 298 bytes


In [15]:
from zipfile import ZipFile, ZIP_DEFLATED

saved_config = json.loads(
    (OUTPUT_DIR / "tokenization_config.json").read_text(encoding="utf-8")
)
saved_manifest = json.loads(
    (OUTPUT_DIR / "split_manifest.json").read_text(encoding="utf-8")
)

assert saved_config == tokenization_config
assert saved_manifest == split_manifest

for filename, expected_report in [
    ("input_lengths.csv", length_report),
    ("tokenization_report.csv", tokenization_report),
    ("batch_report.csv", batch_report)
]:
    pd.testing.assert_frame_equal(
        pd.read_csv(OUTPUT_DIR / filename),
        expected_report,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12
    )

export_files = [
    "tokenization_config.json",
    "split_manifest.json",
    "input_lengths.csv",
    "tokenization_report.csv",
    "batch_report.csv"
]

archive_path = Path("/content/m3_tokenization_setup.zip")

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for filename in export_files:
        archive.write(OUTPUT_DIR / filename, arcname=filename)

print("Configuration, split identifiers, and all reports verified.")
print(f"Archive: {archive_path.name}")
files.download(str(archive_path))

Configuration, split identifiers, and all reports verified.
Archive: m3_tokenization_setup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>